In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [28]:
print("Loading California Housing Dataset...")
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename("HousePrice")], axis=1)

Loading California Housing Dataset...


In [29]:
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [30]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

ridge = Ridge()
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2 = r2_score(y_test, ridge_pred)

In [31]:
print("\n--- Overfitting Detection ---")
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

train_pred = tree.predict(X_train)
test_pred = tree.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print(f"Unconstrained Decision Tree - Train RMSE: {train_rmse:.4f}")
print(f"Unconstrained Decision Tree - Test RMSE: {test_rmse:.4f}")
print("Interpretation: A large gap between training and test RMSE indicates overfitting, common in unconstrained tree-based models.")


--- Overfitting Detection ---
Unconstrained Decision Tree - Train RMSE: 0.0000
Unconstrained Decision Tree - Test RMSE: 0.7030
Interpretation: A large gap between training and test RMSE indicates overfitting, common in unconstrained tree-based models.


In [ ]:
print("\n--- Cross-Validation ---")
cv_scores = cross_val_score(
    tree, X_scaled, y,
    scoring="neg_root_mean_squared_error",
    cv=5
)
cv_rmse = -cv_scores.mean()
print(f"Cross-Validated RMSE (5-Fold): {cv_rmse:.4f}")

print("\n--- Hyperparameter Tuning ---")
param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10]
}

grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5
)

grid.fit(X_train, y_train)
print(f"Best parameters found: {grid.best_params_}")


--- Cross-Validation ---
Cross-Validated RMSE (5-Fold): 0.8957

--- Hyperparameter Tuning ---


In [ ]:
# Step 8: Evaluate Optimized Model
best_tree = grid.best_estimator_
y_pred = best_tree.predict(X_test)

# FIX: Used np.sqrt() instead of squared=False
tuned_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
tuned_r2 = r2_score(y_test, y_pred)

print(f"Optimized Decision Tree - Test RMSE: {tuned_rmse:.4f}")
print(f"Optimized Decision Tree - Test R2: {tuned_r2:.4f}")